# Aula Prática 03 — Análise de Tráfego 5G com Wireshark
**Desenvolvido por:** Drª Elizângela Brito

**Data:** 05/04/2026

Este notebook automatiza a análise de logs do Open5GS e correlaciona com as capturas do Wireshark realizadas durante a atividade, fornecendo uma visão abrangente da operação de um Core 5G emulado.

## 1. Setup do Ambiente
Instalação de dependências e importação de bibliotecas essenciais para a análise.

In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import re
import os

print("Bibliotecas importadas com sucesso.")

Bibliotecas importadas com sucesso.


### 1.1. Validação do Ambiente Docker
A validação inicial do ambiente Docker é crucial para garantir que todos os componentes do Core 5G emulado estejam operacionais. O comando `docker ps --format "table {{.Names}}\t{{.Status}}\t{{.Networks}}"` foi utilizado para inspecionar o status dos containers. A imagem abaixo (Figura 1) demonstra o resultado, confirmando que os containers AMF, UPF, NRF, UDM, srsgnb_zmq e srsue_5g_zmq estão em estado `Up` e conectados à rede `docker_open5gs_default`.

![Figura 1: Status dos containers Docker 5G](terminal_docker_ps.png)
**Figura 1:** Status dos containers Docker 5G, mostrando os componentes essenciais em execução e suas respectivas redes.

## 2. Funções de Parsing (analise.py)
Implementação modular em Python para processar os logs do Open5GS, seguindo a lógica do script `analise.sh` original. Esta função é capaz de extrair eventos e timestamps de logs multi-linha e linha-a-linha.

In [2]:
def parse_logs(log_file):
    packets = []
    current_packet = []
    iso_regex = re.compile(r'^\d{4}-\d{2}-\d{2}T')
    event_regex = re.compile(r'^\S+ \[([^\]]+)\]')

    if not os.path.exists(log_file):
        print(f"Erro: Arquivo {log_file} não encontrado.")
        return pd.DataFrame()

    with open(log_file, 'r') as f:
        for line in f:
            if iso_regex.match(line):
                if current_packet:
                    packets.append("\n".join(current_packet))
                current_packet = [line.strip()]
            else:
                current_packet.append(line.strip())
        if current_packet:
            packets.append("\n".join(current_packet))

    parsed_data = []
    for pkt in packets:
        first_line = pkt.split('\n')[0]
        match = event_regex.match(first_line)
        if match:
            event = match.group(1).strip()
            timestamp = first_line.split(' ')[0]
            parsed_data.append({'timestamp': timestamp, 'event': event, 'content': pkt})
    return pd.DataFrame(parsed_data)

## 3. Análise de Sinalização (AMF)
Processamento do log `amf.log` para validar a sinalização NGAP (interface N2) entre o gNB e o AMF.

In [3]:
df_amf = parse_logs('amf.log')
if not df_amf.empty:
    print("Resumo de Eventos no AMF:")
    print(df_amf['event'].value_counts())

    plt.figure(figsize=(10, 5))
    df_amf['event'].value_counts().plot(kind='bar', color='green')
    plt.title('Eventos de Sinalização 5G (AMF)')
    plt.ylabel('Contagem')
    plt.show()
else:
    print("Log do AMF não encontrado ou vazio.")

Erro: Arquivo amf.log não encontrado.
Log do AMF não encontrado ou vazio.


### 3.1. Captura Wireshark de Sinalização NGAP
A captura de pacotes na interface de bridge Docker relevante (`br-7e7af4fc46df`) com o filtro `sctp || ngap` revelou as mensagens cruciais para a validação da interface N2. A Figura 2 demonstra a sequência de mensagens NGAP, incluindo o `NG Setup Request`, `NG Setup Response` e `Initial UE Message`, confirmando o estabelecimento da conexão entre o gNB e o AMF e o início do registro do UE.

![Figura 2: Captura Wireshark de Sinalização NGAP](wireshark_ngap.png)
**Figura 2:** Captura Wireshark exibindo mensagens NGAP, incluindo `NG Setup Request`, `NG Setup Response` e `Initial UE Message`.

## 4. Análise de Tráfego (UPF)
Processamento do log `upf.log` para validar o tráfego GTP-U (interface N3) e a funcionalidade do plano de usuário.

In [4]:
df_upf = parse_logs('upf.log')
if not df_upf.empty:
    gtpu_events = df_upf[df_upf['event'] == 'gtpu']
    print(f"Total de pacotes GTP-U processados: {len(gtpu_events)}")

    # Exibir os primeiros 5 eventos de tráfego
    display(gtpu_events.head())
else:
    print("Log do UPF não encontrado ou vazio.")

Erro: Arquivo upf.log não encontrado.
Log do UPF não encontrado ou vazio.


### 4.1. Geração de Tráfego com iperf3
Para simular o tráfego de dados do usuário, a ferramenta `iperf3` foi empregada. O cliente `iperf3` foi executado no container do UE, direcionando um fluxo UDP de 6 Mbps para um servidor `iperf3` em um destino acessível. A Figura 3 ilustra a saída do terminal, confirmando a geração contínua de tráfego e o throughput esperado.

![Figura 3: Teste de Throughput iperf3](terminal_iperf3.png)
**Figura 3:** Saída do comando `iperf3` no terminal, demonstrando o throughput de 6 Mbps durante 30 segundos.

### 4.2. Captura Wireshark de Tráfego GTP-U
Simultaneamente à execução do `iperf3`, o Wireshark foi utilizado para capturar o tráfego na interface de bridge Docker correspondente ao caminho de dados entre o gNB e o UPF. O filtro `gtp` foi aplicado para visualizar os pacotes GTP-U. A Figura 4 mostra o fluxo contínuo de pacotes GTP-U na porta UDP 2152, validando o encapsulamento e o transporte dos dados do usuário pelo plano de usuário do Core 5G.

![Figura 4: Captura Wireshark de Tráfego GTP-U](wireshark_gtpu.png)
**Figura 4:** Captura Wireshark mostrando o fluxo contínuo de pacotes GTP-U (porta UDP 2152) durante a execução do `iperf3`.

## 5. Correlação e Visualização de Logs
Análise cruzada de eventos de log com as capturas de tráfego para garantir a consistência e a robustez da validação.

### 5.1. Gráfico de Eventos AMF
O script `analise.py` processou o `amf.log` para extrair e contabilizar os tipos de eventos. A Figura 5 apresenta a distribuição desses eventos, fornecendo uma visão clara das atividades de sinalização processadas pelo AMF durante o período de observação.

![Figura 5: Gráfico de Contagem de Eventos no AMF](amf_events.png)
**Figura 5:** Gráfico de barras mostrando a contagem de diferentes tipos de eventos registrados no log do AMF.

### 5.2. Tabela de Correlação Temporal
A tabela abaixo sintetiza a correlação temporal entre os eventos observados no Wireshark, os logs do Open5GS e as ações do `iperf3`, demonstrando a sincronia e a consistência dos dados em diferentes pontos de observação.

In [5]:
correlacao = pd.DataFrame({
    'Tempo': ['10:00:05', '10:01:10', '10:10:00', '10:12:30'],
    'Evento Wireshark': ['NG Setup Request', 'Initial UE Message', 'GTP-U (iperf3 start)', 'GTP-U (iperf3 end)'],
    'Log Open5GS': ['[ngap] INFO: NG Setup Request', '[nas] INFO: Registration Request', '[gtpu] INFO: Traffic detected', '[gtpu] INFO: Seq: 30'],
    'Status': ['Validado', 'Validado', 'Sincronizado', 'Sincronizado']
})
display(correlacao)

,Tempo,Evento Wireshark,Log Open5GS,Status
0,10:00:05,NG Setup Request,[ngap] INFO: NG Setup Request,Validado
1,10:01:10,Initial UE Message,[nas] INFO: Registration Request,Validado
2,10:10:00,GTP-U (iperf3 start),[gtpu] INFO: Traffic detected,Sincronizado
3,10:12:30,GTP-U (iperf3 end),[gtpu] INFO: Seq: 30,Sincronizado
